# Language-Model Adapter — DIMER Artifact Inference Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_artifact_inference_colab.ipynb)

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification `1.0`

This notebook consumes an **externally supplied PEFT adapter ZIP**. No training or fine-tuning occurs. It validates archive structure and manifest integrity, checks canonical model and runtime provenance, reconstructs through the production inference surface, accepts a real new prompt, and writes machine-readable results.

It does **not** establish sender authenticity, benchmark accuracy, factual correctness, safety, fairness, calibration, robustness, or production fitness.


## Prerequisites and trust boundary

- **Runtime:** Google Colab with a CUDA GPU.
- **Private production source:** add a Colab Secret `GITHUB_TOKEN` with read access to the private `language-model-finetuner` repository and grant this notebook access. The token is used only through an ephemeral Git auth header; it is not printed, embedded in the clone URL, or stored in repository config.
- **Artifact:** upload exactly one ZIP produced by the E2E notebook/production artifact contract.

Manifest consistency is not sender authenticity. A whole-ZIP SHA-256 is an authenticity aid only when its expected value comes through an independently trusted channel. Safe extraction is not permission to deserialize arbitrary executable formats; this path requires safetensors/JSON and `trustRemoteCode=false`.


## 1. Install exact dependencies and immutable production source

The same public pipeline-support and private finetuner runtime revisions used by the E2E producer are loaded here and recorded separately from the notebook/PR head.


In [ ]:
%pip -q install transformers==5.16.1 tokenizers==0.23.2 huggingface-hub==1.30.0 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.49.0 safetensors==0.8.0 datasets==4.8.5 pandas==2.3.3 PyYAML==6.0.3 Jinja2==3.1.6
%pip -q install --no-deps git+https://github.com/kurtvalcorza/language-model-pipeline.git@afaf1f032cd7e9751db1ee8eb71b542f9bd0b15f


In [ ]:
import json
import sys
from pathlib import Path

from lmpipeline.tutorial_runtime import (
    checkout_private_finetuner,
    github_token_from_runtime,
)

PIPELINE_RUNTIME_REVISION = "afaf1f032cd7e9751db1ee8eb71b542f9bd0b15f"
FINETUNER_RUNTIME_REVISION = "ecbab8cfbb95c061235f68c087141ab7af462b9d"
FINETUNER_ROOT = Path("/content/language-model-finetuner")

_GITHUB_TOKEN = github_token_from_runtime()
checkout_private_finetuner(
    FINETUNER_ROOT,
    revision=FINETUNER_RUNTIME_REVISION,
    token=_GITHUB_TOKEN,
)
del _GITHUB_TOKEN
sys.path.insert(0, str(FINETUNER_ROOT / "src"))


In [ ]:
import pandas as pd
import torch

from lmpipeline.tutorial_api import (
    assert_finetuner_checkout,
    assert_runtime_compatible,
    assert_tutorial_runtime,
    resolve_artifact_model,
    sha256_file,
    validate_prompt,
)
from lmpipeline.tutorial_runtime import consume_adapter_archive
from finetuner.inference import (
    generate_reply,
    load_adapter_for_inference,
    verify_adapter_active,
)

RUNTIME = assert_tutorial_runtime()
assert_finetuner_checkout(FINETUNER_ROOT, FINETUNER_RUNTIME_REVISION)
if not torch.cuda.is_available():
    raise RuntimeError("Artifact inference requires a CUDA GPU.")
print(json.dumps({
    "pipelineRuntimeRevision": PIPELINE_RUNTIME_REVISION,
    "finetunerRuntimeRevision": FINETUNER_RUNTIME_REVISION,
    "runtime": RUNTIME,
}, indent=2))


## 2. Upload and validate the external adapter ZIP

The strict consumer requires `artifact-manifest.json` at the ZIP root. This closes the sibling-file ambiguity of accepting a valid manifested subdirectory alongside unmanifested content. Path traversal, backslashes, symlinks, duplicates, excessive expansion, hash/size mismatches, missing required files, and unlisted files are rejected before model state is loaded.


In [ ]:
EXPECTED_ARTIFACT_ZIP_SHA256 = "" # @param {type:"string"}

from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one adapter ZIP")
name, payload = next(iter(uploaded.items()))
if not name.lower().endswith(".zip"):
    raise ValueError("The external artifact must be a .zip archive")
ARCHIVE = Path("/content") / Path(name).name
ARCHIVE.write_bytes(payload)

ARTIFACT_ROOT, MANIFEST, PROVENANCE = consume_adapter_archive(
    ARCHIVE,
    extraction_root="/content/dimer-language-model-artifact",
    expected_archive_sha256=EXPECTED_ARTIFACT_ZIP_SHA256,
)
print({
    "archive": ARCHIVE.name,
    "archiveSha256": sha256_file(ARCHIVE),
    "manifestedFiles": len(MANIFEST["files"]),
    "totalBytes": MANIFEST["totalBytes"],
})


## 3. Verify canonical model identity and producer/consumer compatibility

The artifact must match the canonical model registry, exact base revision, critical package versions, and both runtime-source revisions. This check includes `safetensors`, which the producer records in `packageVersions`.


In [ ]:
ENTRY = resolve_artifact_model(PROVENANCE)
assert_runtime_compatible(PROVENANCE, RUNTIME)

RECORDED_RUNTIME_REVISIONS = PROVENANCE.get("runtimeRevisions") or {}
if RECORDED_RUNTIME_REVISIONS.get("pipeline") != PIPELINE_RUNTIME_REVISION:
    raise ValueError("Artifact was produced with a different pipeline runtime revision")
if RECORDED_RUNTIME_REVISIONS.get("finetuner") != FINETUNER_RUNTIME_REVISION:
    raise ValueError("Artifact was produced with a different finetuner runtime revision")

print(json.dumps({
    "modelKey": ENTRY.key,
    "baseModel": ENTRY.model_id,
    "baseModelRevision": ENTRY.revision,
    "datasetDigest": PROVENANCE.get("datasetDigest"),
    "packageVersions": PROVENANCE.get("packageVersions"),
    "runtimeRevisions": RECORDED_RUNTIME_REVISIONS,
}, indent=2))


## 4. Reconstruct through the production inference surface

The exact base revision and serialized PEFT adapter are loaded through `finetuner.inference`. Adapter activity is proven numerically using non-zero LoRA B weights and a non-zero adapter-on/off logit delta.


In [ ]:
MODEL, TOKENIZER = load_adapter_for_inference(
    ARTIFACT_ROOT,
    entry=ENTRY,
    device="cuda",
    quantized=True,
)
ACTIVITY = verify_adapter_active(
    MODEL,
    TOKENIZER,
    prompt="Kumusta.",
)
print({
    "baseModel": ENTRY.model_id,
    "revision": ENTRY.revision,
    "adapterActivity": ACTIVITY,
    "gpuMemoryGiB": round(torch.cuda.memory_allocated() / 1024**3, 2),
})


## 5. Validate real new user input

Edit `CUSTOM_PROMPT`. Rendered prompt tokens plus requested output budget must remain within the training-time sequence ceiling recorded in provenance.


In [ ]:
CUSTOM_PROMPT = "Sumulat ng dalawang pangungusap tungkol sa responsableng paggamit ng AI." # @param {type:"string"}
MAX_NEW_TOKENS = 128 # @param {type:"integer"}

TRAINING = (PROVENANCE.get("job") or {}).get("training") or {}
MAX_SEQUENCE_LENGTH = int(TRAINING.get("maxSequenceLength", 0))
if MAX_SEQUENCE_LENGTH <= 0 or MAX_SEQUENCE_LENGTH > int(ENTRY.max_sequence_length):
    raise ValueError("Artifact provenance has an invalid maxSequenceLength")
PROMPT_TOKENS = validate_prompt(
    TOKENIZER,
    CUSTOM_PROMPT,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
)
if MAX_NEW_TOKENS <= 0 or PROMPT_TOKENS + MAX_NEW_TOKENS > MAX_SEQUENCE_LENGTH:
    raise ValueError(
        f"prompt ({PROMPT_TOKENS}) + MAX_NEW_TOKENS ({MAX_NEW_TOKENS}) exceeds "
        f"the {MAX_SEQUENCE_LENGTH}-token context ceiling"
    )
print({
    "promptTokens": PROMPT_TOKENS,
    "maxNewTokens": MAX_NEW_TOKENS,
    "contextCeiling": MAX_SEQUENCE_LENGTH,
})


## 6. Generate and export machine-readable results

Greedy decoding is the reproducible verification path. Prediction JSONL contains the prompt and generated text; treat it as sensitive when the input is sensitive.


In [ ]:
DECODING = {"do_sample": False}
ANSWER = generate_reply(
    MODEL,
    TOKENIZER,
    CUSTOM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    decoding=DECODING,
)
if not ANSWER:
    raise RuntimeError("The reconstructed adapter generated an empty response")

RESULT = {
    "inputId": "prompt-1",
    "prompt": CUSTOM_PROMPT,
    "promptTokens": PROMPT_TOKENS,
    "output": ANSWER,
    "modelId": ENTRY.model_id,
    "modelRevision": ENTRY.revision,
    "decoding": {"doSample": False, "maxNewTokens": MAX_NEW_TOKENS},
}
OUTPUT_JSONL = Path("/content/artifact_inference_predictions.jsonl")
OUTPUT_JSONL.write_text(
    json.dumps(RESULT, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
OUTPUT_PROVENANCE = Path("/content/artifact_inference_provenance.json")
OUTPUT_PROVENANCE.write_text(
    json.dumps({
        "profile": "ARTIFACT-INFERENCE",
        "notebookSpecVersion": "1.0",
        "artifactSha256": sha256_file(ARCHIVE),
        "baseModel": ENTRY.model_id,
        "baseModelRevision": ENTRY.revision,
        "pipelineRuntimeRevision": PIPELINE_RUNTIME_REVISION,
        "finetunerRuntimeRevision": FINETUNER_RUNTIME_REVISION,
        "runtime": RUNTIME,
        "adapterActivity": ACTIVITY,
        "decoding": RESULT["decoding"],
    }, indent=2),
    encoding="utf-8",
)
display(pd.DataFrame([RESULT])[["inputId", "prompt", "output"]])
print("wrote", OUTPUT_JSONL, "and", OUTPUT_PROVENANCE)


## Interpretation and release boundary

A successful run establishes archive/manifest validation, canonical provenance parity, compatible producer/consumer runtime packages, production reconstruction, active serialized LoRA deltas, context validation, generation, and machine-readable output.

It does **not** establish sender authenticity or release-grade status. Release-grade status additionally requires the clean supported-runtime record tied to the candidate notebook/PR head and both immutable runtime-source revisions.
